In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import  roc_auc_score

In [15]:
X_train = pd.read_pickle('../datos/entrenamiento/X_train.pkl')
X_test = pd.read_pickle('../datos/entrenamiento/X_test.pkl')
y_train = pd.read_pickle('../datos/entrenamiento/y_train.pkl')
y_test = pd.read_pickle('../datos/entrenamiento/y_test.pkl')

In [ ]:
algoritmo = XGBClassifier()
grid = {
    'n_estimators': [100, 200, 300],         
    'learning_rate': [0.01, 0.05, 0.1],       
    'max_depth': [3, 5, 7],                   
    'subsample': [0.7, 0.8, 1.0],             
    'colsample_bytree': [0.7, 0.8, 1.0],     
    'reg_alpha': [0, 0.1, 1],                 
    'reg_lambda': [1, 5, 10]
}

In [17]:
from sklearn.model_selection import RandomizedSearchCV

grid_search = RandomizedSearchCV(algoritmo, 
                                 grid, 
                                 cv = 5, 
                                 scoring = 'roc_auc',
                                 n_jobs = -1,
                                 verbose = 2)

In [18]:
mejor_modelo = grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [19]:
print('Mejor combinación:' , grid_search.best_params_)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.DataFrame(grid_search.cv_results_)[['params', 'mean_test_score']].sort_values('mean_test_score', ascending = False)

Mejor combinación: {'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 1, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


,params,mean_test_score
7,"{'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 1, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}",0.936850
3,"{'subsample': 0.7, 'reg_lambda': 5, 'reg_alpha': 1, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.7}",0.936696
6,"{'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",0.935922
4,"{'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 0.1, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}",0.933359
1,"{'subsample': 0.7, 'reg_lambda': 5, 'reg_alpha': 0.1, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}",0.930214
5,"{'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}",0.928293
8,"{'subsample': 0.7, 'reg_lambda': 5, 'reg_alpha': 0.1, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.7}",0.919243
2,"{'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.7}",0.912694
9,"{'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.1, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 1.0}",0.911192
0,"{'subsample': 1.0, 'reg_lambda': 5, 'reg_alpha': 0, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.7}",0.891811


In [24]:
xgb = XGBClassifier(subsample=0.8, reg_lambda=5, reg_alpha=1, n_estimators=300, max_depth=7, learning_rate=0.05, colsample_bytree=0.8)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, xgb_pred))

0.9348965838739194


In [25]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metricas_cv = cross_val_score(estimator=xgb, 
                              X=X_train, 
                              y=y_train, 
                              cv=skf, 
                              scoring='roc_auc')
print("ROC AUC Score (Cross-Validation):", metricas_cv.mean())

ROC AUC Score (Cross-Validation): 0.9361114600748938
